# Artificial coupling between the pre- and post-stimulus windows

PRIME's premise is that the pre-stimulus brain state predicts the post-stimulus TEP. That premise is
only testable if the preprocessing does not *itself* create a link between the two windows. This
notebook replays the intervention blocks offline from the continuous NeurOne recordings, using the
session's own calibration bundle and dipole fit, and asks two things:

1. **Is there a mechanical pre→post link in the current pipeline, and which step creates it?**
2. **Does correcting the post-stimulus dipole amplitude by a pre-stimulus floor remove an artifact,
   or remove the signal PRIME depends on?**

Those are different questions and the second is only meaningful once the first is answered — which
is why the diagnostics in sections 10 and 11 are not optional extras.

## Four candidate coupling mechanisms, read out of the live config

- **`baseline = [-0.025, -0.015]`** — a 10-sample mean, subtracted from *every* sample of the epoch
  including the 38-50 ms dipole window, and applied **twice** (before and after ICA). A 10-sample
  mean is noisy, so this injects pre-stimulus fluctuation straight into the post-stimulus amplitude
  with a negative sign. Strongest a-priori candidate, and the one this notebook's Arm B targets.
- **`filter_opts = [2, 47] Hz, order 2`, applied with `filtfilt`** — zero-phase, therefore
  non-causal. A 2 Hz high-pass run forward-and-backward spreads energy in *both* directions on a
  scale of tens of ms, so TMS-artifact residue leaks backwards into the "pre-stimulus" window and
  pre-stimulus content leaks forwards into the dipole window.
- **`artifact_window_1 = [-0.014, +0.014]`, interpolated** — literally spans the stimulus, blending
  pre- and post-stimulus samples by construction before anything else runs.
- **`reject_range = [0.02, 0.06]`** overlaps the dipole window [0.038, 0.050], so trial rejection is
  decided on the same samples the amplitude is measured from — a selection effect stacked on top.

## Two defects found by auditing this notebook against pilot003 and pilot004

**`csv_match` cannot be trusted.** `organise_neurone.m` flagged 162 of pilot003's 925 triggers as
matched, and 158 of those 162 point at the wrong trial; pilot004 is clean at 925/925. The recording
itself is fine — triggers and CSV rows are both in time order and the same length, and
`onset - pulse_time` is constant to 0.00 ms with inter-stimulus intervals agreeing to 0.20 ms. So
events are paired positionally and the pairing is verified, not assumed. Reading `csv_match` instead
is what made pilot003's replication gate fail at corr 0.03.

**Pre-stimulus chunks were not baselined.** An average reference removes what is common across
channels, not each channel's own DC, so a pre-stimulus chunk that skips baseline #1 carries the raw
~1.3e4 offset into the dipole fit and comes out around a thousand times a post-stimulus amplitude.
Every pre/post comparison built on it — the floor, the percentile, the lag sweep, the ablation
reference — was comparing DC drift. Each chunk now gets the same local baseline the online epoch
gets, taken 16-26 ms before the window it is fitted in, which is the geometry `apply_baseline`
produces for a post epoch.

## One structural finding that changes the plan

The post epoch is cropped to `[post_initial_tmin, post_initial_tmax]` = **[-30, +100] ms as the very
first step**, before either baseline correction. **A 1 s baseline does not fit inside it.** Simply
widening the epoch is not a safe fix either: the calibration bundle stores the ocular check's
`time_indices_of_interest` as *sample indices* into the calibrated epoch geometry, so changing the
epoch bounds silently points them at the wrong samples.

So Arm B keeps the epoch geometry byte-identical and changes only the *value* subtracted: the robust
centre is computed from a separate long pre-stimulus window and subtracted as a per-channel constant
at exactly the two points `apply_baseline` would have acted. Trial-rejection decisions are inherited
from Arm A, so the two arms differ in one thing only.

## 1 · Setup

Inputs per session: the continuous `task-intervention-all` EEGLAB pair written by
`organise_neurone.m`, its `events.tsv` (already joined to the trials CSV — `onset` is the epoching
anchor and every online per-trial value rides along), plus the session's `calibration_bunble.npy`
and `dipole_fitting_info.npy`.

The `.fdt` is several GB, so it is memory-mapped and sliced per trial rather than loaded.

In [ ]:
from __future__ import annotations

import os, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.stats import median_abs_deviation, trim_mean
import matplotlib.pyplot as plt
import mne

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 100, "figure.figsize": (10, 3.4),
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
})

COLOR_PREDETERMINED = "#5b8ff9"   # blue
COLOR_PRIME_SINGLE  = "#e8684a"   # orange


def _find_loop_root() -> Path:
    env = os.environ.get("LOOP_ROOT")
    candidates = [env] if env else []
    candidates += [r"D:\Linus\Loop", str(Path.home() / "Desktop" / "Loop")]
    for c in candidates:
        if c and (Path(c) / "BIDS").is_dir() and (Path(c) / "prime").is_dir():
            return Path(c)
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / "BIDS").is_dir() and (d / "prime").is_dir():
            return d
    raise FileNotFoundError("Could not locate the Loop root; set the LOOP_ROOT environment variable.")


LOOP_ROOT = _find_loop_root()
sys.path.insert(0, str(LOOP_ROOT / "prime" / "decider"))

from prime_core.preprocessing.preprocessor import (
    Preprocessor, crop_eeg_buffer, resample_buffer_polyphase,
    _numpy_to_epochs_array, _interpolate_bad_channels,
)
from prime_core.preprocessing.dipole_fitter import (
    DipoleFitter, COMMON_CHANNELS, fit_dipole_to_single_trial,
)
from prime_core.preprocessing.utils.ssp_sir_python import ssp_sir_single_trial
from prime_core.preprocessing.config import get_default_config
from prime_core.prime_config import (
    get_raw_sfreq, get_post_initial_time_range, get_post_time_range,
    get_dipole_time_range, get_processed_sfreq, epoch_n_times,
)

FORWARD_PATH = LOOP_ROOT / "prime" / "offline_data" / "fsaverage" / "fsaverage-fwd.fif"
RAW_SFREQ = get_raw_sfreq()
PROC_SFREQ = get_processed_sfreq()
POST_I_TMIN, POST_I_TMAX = get_post_initial_time_range()
POST_TMIN, POST_TMAX = get_post_time_range()
DIPOLE_TMIN, DIPOLE_TMAX = get_dipole_time_range()
CFG = get_default_config()

SESSIONS_ALL = {
    "pilot003": LOOP_ROOT / "BIDS" / "sub-pilot003" / "ses-prime" / "103" / "prime",
    "pilot004": LOOP_ROOT / "BIDS" / "sub-pilot004" / "ses-prime" / "104" / "prime",
    "pilot005": LOOP_ROOT / "BIDS" / "sub-pilot005" / "ses-prime" / "105" / "prime",
}
RUN_SESSIONS = ("pilot004", "pilot005")          # sessions this run analyses
# pilot003's intervention export is usable (its csv_match is broken, but align_events repairs it);
# it is left out here because 004 and 005 are the two clean sessions. Add it back to the tuple.
SESSIONS = {k: v for k, v in SESSIONS_ALL.items() if k in RUN_SESSIONS}
EEG_DIRS = {n: d.parent.parent / "eeg" for n, d in SESSIONS.items()}
BUNDLE_NAMES = ("calibration_bunble.npy", "calibration_bundle.npy")

# ---- the knobs this notebook varies ----
# The classifier's own feature window and the stimulus-artifact taper bracket the baseline from
# both sides; every candidate is scored against both in the print below.
MODEL_TMIN, MODEL_TMAX = -0.065, -0.015          # prime.yaml `model`
TAPER_TMIN = CFG.artifact_window_1[0]            # fix_stim_artifact starts here

BASELINE_WINDOWS = {                             # seconds relative to the pulse
    "online [-25,-15]":  (-0.025, -0.015),
    "late [-15,-5]":     (-0.015, -0.005),
    "epoch [-30,-15]":   (-0.030, -0.015),
    "late [-15,-10]":    (-0.015, -0.010),
    "late [-20,-10]":    (-0.020, -0.010),
    "wide [-65,-15]":    (-0.065, -0.015),
    "early [-65,-55]":   (-0.065, -0.055),
    "clear [-105,-95]":  (-0.105, -0.095),
    "clear [-205,-105]": (-0.205, -0.105),
    "long [-250,-15]":   (-0.250, -0.015),
    "long [-1000,-15]":  (-1.000, -0.015),
}
BASELINE_CENTRE = "mean"            # estimator over BASELINE_WINDOWS; "mean" = apply_baseline
BASELINE_SWEEP_LIMIT = 60           # trials per session in the sweep; None = all
ARM_B_WINDOW   = (-1.000, -0.015)   # window Arm B's robust centre uses in sections 7-10
LONG_BASELINE  = ARM_B_WINDOW       # backwards-compatible alias
ROBUST_CENTRE  = "median"           # "median" | "trimmed" | "mean"
PRE_FLOOR_SPAN = (-1.000, -0.030)   # extent tiled by pre_floor; independent of the baseline knob
PRE_CHUNK_STEP_MS = 10              # stride of the pre-stimulus floor windows
PROBE_LAG_MS = -35                  # fixed leakage probe used by the baseline sweep

print(f"Loop root : {LOOP_ROOT}")
print(f"raw {RAW_SFREQ:.0f} Hz -> processed {PROC_SFREQ:.0f} Hz")
print(f"post epoch fed to the pipeline : [{POST_I_TMIN*1000:.0f}, {POST_I_TMAX*1000:.0f}] ms "
      f"-> cropped to [{POST_TMIN*1000:.0f}, {POST_TMAX*1000:.0f}] ms")
print(f"dipole window                  : [{DIPOLE_TMIN*1000:.0f}, {DIPOLE_TMAX*1000:.0f}] ms "
      f"({epoch_n_times(DIPOLE_TMIN, DIPOLE_TMAX, PROC_SFREQ)} samples)")
print(f"baseline (online)              : [{CFG.baseline[0]*1000:.0f}, {CFG.baseline[1]*1000:.0f}] ms "
      f"= {round((CFG.baseline[1]-CFG.baseline[0])*PROC_SFREQ)} samples, applied twice")
print(f"baseline (Arm B)               : [{LONG_BASELINE[0]*1000:.0f}, {LONG_BASELINE[1]*1000:.0f}] ms "
      f"= {round((LONG_BASELINE[1]-LONG_BASELINE[0])*PROC_SFREQ)} samples, {ROBUST_CENTRE}")
print(f"filter                         : {CFG.filter_opts.cutoff} Hz {CFG.filter_opts.btype}, "
      f"order {CFG.filter_opts.order} (filtfilt -> non-causal)")
print(f"artifact_window_1 (interpolated): [{CFG.artifact_window_1[0]*1000:.0f}, "
      f"{CFG.artifact_window_1[1]*1000:.0f}] ms  -- spans the stimulus")
print(f"reject_range                   : [{CFG.reject_range[0]*1000:.0f}, "
      f"{CFG.reject_range[1]*1000:.0f}] ms  -- overlaps the dipole window")
print("\ncandidate baseline windows -- three hard constraints:")
print(f"  the classifier's model window is [{MODEL_TMIN*1000:.0f}, {MODEL_TMAX*1000:.0f}] ms; a "
      f"baseline inside it is subtracted from the epoch it is asked to predict")
print(f"  fix_stim_artifact tapers everything after {TAPER_TMIN*1000:.0f} ms toward zero")
print(f"  the post epoch is only [{POST_I_TMIN*1000:.0f}, {POST_I_TMAX*1000:.0f}] ms, so anything "
      f"earlier than {POST_I_TMIN*1000:.0f} ms cannot go through apply_baseline at all")
print(f"  {'window':20s} {'samples':>8s} {'in model':>9s} {'in taper':>9s}  {'route':9s} verdict")
for _lbl, (_a, _b) in BASELINE_WINDOWS.items():
    _n = round((_b - _a) * PROC_SFREQ)
    _ov_m = max(0.0, min(_b, MODEL_TMAX) - max(_a, MODEL_TMIN)) / (_b - _a)
    _ov_t = max(0.0, _b - max(_a, TAPER_TMIN)) / (_b - _a)
    _route = "in-epoch" if _a >= POST_I_TMIN else "block"
    _v = ("inside the taper"    if _ov_t > 0.5 else
          "leaks into features" if _ov_m > 0.5 else
          "partial overlap"     if _ov_m > 0 or _ov_t > 0 else "clean")
    print(f"  {_lbl:20s} {_n:8d} {100*_ov_m:8.0f}% {100*_ov_t:8.0f}%  {_route:9s} {_v}")


# ---------------------------------------------------------------- never report a silent blank
MIN_N = 4          # smallest n a correlation is reported for


def safe_corr(x, y, *, min_n=MIN_N):
    """Pearson r plus the reason it is missing, so a blank cell always explains itself."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    n = int(m.sum())
    if n < min_n:
        return np.nan, n, f"only {n} finite pairs (need {min_n})"
    if np.std(x[m]) == 0 or np.std(y[m]) == 0:
        return np.nan, n, "one side is constant"
    return float(np.corrcoef(x[m], y[m])[0, 1]), n, ""


## 2 · The continuous recording and its events

`organise_neurone.m` writes an EEGLAB `.set`/`.fdt` pair plus an `events.tsv` whose rows are already
matched to `trials_intervention.csv` (`csv_match`). The first 60 channels are exactly
`COMMON_CHANNELS` in the order the forward model expects, followed by the two EMG channels, and the
sample rate is the pipeline's own `raw_sfreq` — so no resampling or reordering is needed before
handing data to `Preprocessor`. All of that is asserted rather than assumed.

In [ ]:
STAGE_ORDER = ["calibration", "intervention_block_1", "intervention_block_2",
               "intervention_block_3", "intervention_block_4"]


def align_events(name: str, events: pd.DataFrame, iv: pd.DataFrame) -> pd.DataFrame:
    """Pair NeurOne stimulation triggers with `trials_intervention.csv` rows.

    `organise_neurone.m` writes a `csv_match` column, but it cannot be trusted: in pilot003 it
    flags 162 of 925 rows, and 158 of those 162 point at the wrong trial. The recording itself is
    intact -- the triggers and the CSV are both in time order and the same length -- so a
    positional join is exact. That is verified here rather than assumed, by two independent
    checks: `onset - pulse_time` must be a constant with no drift, and the inter-stimulus
    intervals must agree to well under one sample. If the counts differ the function falls back
    to `csv_match` and says so, because then the positional assumption is not safe.
    """
    ev = events.sort_values("onset").reset_index(drop=True)
    iv = iv.loc[:, ~iv.columns.str.startswith("Unnamed")].copy()
    iv["_stage_order"] = iv.stage.map({s: i for i, s in enumerate(STAGE_ORDER)})
    iv = iv.sort_values(["_stage_order", "trial_in_stage"]).reset_index(drop=True)

    if len(ev) != len(iv):
        n_flag = int((ev["csv_match"] == 1).sum()) if "csv_match" in ev else 0
        print(f"   {name}: {len(ev)} triggers vs {len(iv)} CSV trials -- positional join unsafe, "
              f"falling back to csv_match ({n_flag} rows)")
        return (ev[ev.csv_match == 1] if "csv_match" in ev else ev).reset_index(drop=True)

    off = ev.onset.to_numpy() - iv.pulse_time.to_numpy()
    drift_ms = 1000.0 * float(np.nanmax(off) - np.nanmin(off))
    isi_err = np.abs(np.diff(ev.onset.to_numpy()) - np.diff(iv.pulse_time.to_numpy())) * 1000.0
    isi_max = float(np.nanmax(isi_err))
    one_sample_ms = 1000.0 / RAW_SFREQ

    out = iv.drop(columns=["_stage_order"])
    out.insert(0, "onset", ev.onset.to_numpy())
    flagged = int((ev["csv_match"] == 1).sum()) if "csv_match" in ev else 0
    if flagged:
        same = ((ev.loc[ev.csv_match == 1, "stage"].to_numpy()
                 == out.loc[(ev.csv_match == 1).to_numpy(), "stage"].to_numpy())
                & (ev.loc[ev.csv_match == 1, "trial_in_stage"].to_numpy()
                   == out.loc[(ev.csv_match == 1).to_numpy(), "trial_in_stage"].to_numpy()))
        agree = int(same.sum())
    else:
        agree = 0
    ok = drift_ms < 5.0 and isi_max < 2 * one_sample_ms
    print(f"   {name}: positional join of {len(out)} triggers -- offset {off[0]:+.3f} s, "
          f"drift {drift_ms:.2f} ms, max ISI error {isi_max:.2f} ms "
          f"({one_sample_ms:.2f} ms = 1 sample) -> {'VERIFIED' if ok else 'CHECK FAILED'}")
    if flagged:
        print(f"      organise_neurone.m flagged {flagged} of {len(ev)} rows as matched; "
              f"{agree} of those agree with the positional pairing")
    if not ok:
        raise AssertionError(f"{name}: positional alignment failed its own checks; do not proceed")
    return out


def load_recording(name: str) -> dict:
    eeg_dir = EEG_DIRS[name]
    sets = sorted(eeg_dir.glob("*task-intervention-all*_eeg.set"))
    evs = sorted(eeg_dir.glob("*task-intervention-all*_events.tsv"))
    if not sets or not evs:
        raise FileNotFoundError(
            f"{name}: need a task-intervention-all .set/.fdt/_events.tsv triple in {eeg_dir}. "
            f"Found sets={[p.name for p in sets]} events={[p.name for p in evs]}"
        )
    hdr = sio.loadmat(sets[0], squeeze_me=True, struct_as_record=False)
    srate, nbchan, pnts = float(hdr["srate"]), int(hdr["nbchan"]), int(hdr["pnts"])
    ch_names = [c.labels for c in np.atleast_1d(hdr["chanlocs"])]
    fdt = eeg_dir / str(hdr["datfile"])
    if not fdt.is_file():
        fdt = sets[0].with_suffix(".fdt")

    assert srate == RAW_SFREQ, f"{name}: recording is {srate} Hz, pipeline expects {RAW_SFREQ} Hz"
    assert ch_names[:len(COMMON_CHANNELS)] == COMMON_CHANNELS, (
        f"{name}: first {len(COMMON_CHANNELS)} channels are not COMMON_CHANNELS in order; "
        f"got {ch_names[:6]}..."
    )
    # EEGLAB writes EEG.data ([nbchan x pnts], column-major), i.e. all channels per sample in turn.
    data = np.memmap(fdt, dtype="float32", mode="r", shape=(pnts, nbchan)).T

    raw_events = pd.read_csv(evs[0], sep="\t")
    assert {"onset"} <= set(raw_events.columns)
    iv = pd.read_csv(SESSIONS[name] / "trials_intervention.csv")
    events = align_events(name, raw_events, iv)
    return {"data": data, "sfreq": srate, "ch_names": ch_names, "pnts": pnts,
            "events": events.reset_index(drop=True), "set": sets[0], "fdt": fdt}

REC = {}
for name in SESSIONS:
    try:
        REC[name] = load_recording(name)
        r = REC[name]
        print(f"{name}: {r['pnts']} samples ({r['pnts']/r['sfreq']/60:.1f} min), "
              f"{len(r['ch_names'])} channels, {len(r['events'])} trials")
        print(f"   conditions: {dict(r['events'].condition.value_counts(dropna=False))}")
    except FileNotFoundError as e:
        print(f"{name}: NOT AVAILABLE -- {e}")
    except Exception as e:                      # alignment checks raise; one bad session
        print(f"{name}: FAILED TO LOAD -- {type(e).__name__}: {e}")   # must not empty the rest

In [ ]:
# ---------------------------------------------------------------- epoching, and a landmark check
def epoch_raw(rec: dict, onset_s: float, tmin: float, tmax: float,
              *, shift: int | None = None) -> np.ndarray | None:
    '''(n_samples, 60) raw slice around a pulse, at raw_sfreq. None if it runs off either end.

    `rec["onset_shift"]` is a whole-sample correction applied to every slice, calibrated by the
    replication gate in section 4. The events.tsv onsets are quantised to the recording grid and
    can sit a sample away from the sample the online buffer began at; a 0.2 ms offset is invisible
    in the landmark check but costs ~1% of correlation against the saved epochs.
    '''
    sh = int(rec.get("onset_shift", 0)) if shift is None else int(shift)
    i0 = int(round((onset_s + tmin) * rec["sfreq"])) + sh
    i1 = i0 + int(round((tmax - tmin) * rec["sfreq"])) + 1
    if i0 < 0 or i1 > rec["pnts"]:
        return None
    return np.asarray(rec["data"][:len(COMMON_CHANNELS), i0:i1], dtype=np.float64).T


# The stimulus artifact is enormous; it must sit at t=0 of every epoch. If the .fdt were read with
# the wrong memory order, or the onsets were referenced to something other than the recording start,
# this is where it shows.
for name, rec in REC.items():
    ev = rec["events"]
    probe = ev[ev.condition == "predetermined_single"].head(40)
    peaks = []
    for onset in probe.onset:
        e = epoch_raw(rec, float(onset), -0.05, 0.05)
        if e is None:
            continue
        t = np.linspace(-50, 50, e.shape[0])
        peaks.append(t[np.argmax(np.abs(e).mean(axis=1))])
    peaks = np.asarray(peaks)
    print(f"{name}: peak |EEG| within +/-50 ms of onset -- median {np.median(peaks):+.2f} ms, "
          f"IQR [{np.percentile(peaks,25):+.2f}, {np.percentile(peaks,75):+.2f}] "
          f"(n={len(peaks)})  -> {'aligned' if abs(np.median(peaks)) <= 2 else 'MISALIGNED, stop here'}")

## 3 · Restoring the session's own calibration and dipole

Same restoration the other notebooks use: `Preprocessor.from_bundle()` plus a `DipoleFitter` with
`_fitting_info` assigned directly. For a session whose `dipole_fitting_info.npy` was never saved,
recover it first with `replay_dipole_fit.ipynb`.

In [ ]:
def find_bundle(d: Path) -> Path | None:
    for n in BUNDLE_NAMES:
        if (d / n).is_file():
            return d / n
    return None


STATE = {}
for name, d in SESSIONS.items():
    if name not in REC:
        continue
    bp, dp = find_bundle(d), d / "dipole_fitting_info.npy"
    if bp is None or not dp.is_file():
        print(f"{name}: SKIPPED -- bundle={bp is not None}, dipole_fitting_info={dp.is_file()}")
        continue
    bundle = np.load(bp, allow_pickle=True).item()
    pre = Preprocessor.from_bundle(bundle, FORWARD_PATH)
    pre.keep_full_epochs = True
    fitter = DipoleFitter(FORWARD_PATH)
    fitter._fitting_info = np.load(dp, allow_pickle=True).item()
    STATE[name] = {"bundle": bundle, "pre": pre, "fitter": fitter,
                   "IV": pd.read_csv(d / "trials_intervention.csv")}
    fi = fitter._fitting_info
    print(f"{name}: bundle={bp.name}, bad_channels={bundle['bad_channels']}, "
          f"dipole vertex={fi['position_index']}, index_range={tuple(fi['index_range'])}")

## 4 · Stage 0 — the replication gate

Nothing below is interpretable unless the offline replay reproduces the online pipeline. The
tightest available check is not the scalar amplitude but the epoch itself: `<stem>_post_proc.npy`
*is* the online `last_post_full`, the [1, 100] ms epoch after SOUND, SSP-SIR and the second artifact
interpolation. Feeding the same trial's raw slice to the same `Preprocessor` must reproduce it.

Three failure modes, and how to tell them apart:

- **A scale factor** between the `.fdt` values and the online stream shows up as correlation
  ≈ 1.0000 with slope ≠ 1. Slope is therefore reported separately from correlation, and the two are
  judged separately.
- **A whole-sample timing offset.** The events.tsv onsets are quantised to the recording grid and
  can land a sample away from where the online buffer began. The epoch is band-limited to ~500 Hz by
  the polyphase resample, so one raw sample — 0.2 ms — costs about 1% of correlation at 100 Hz while
  leaving slope near 1. The gate searches ±6 samples, takes the median best lag, and applies it as
  `rec["onset_shift"]` before judging anything.
- **Anything else**, which shows as a correlation that does not recover at any lag.

The verdict has three levels, because they license different things. **PASSED** (corr > 0.999)
means the replay is the online epoch. **USABLE** (corr ≥ 0.99, slope within 1%) means a sub-sample
residual remains: every arm and every candidate baseline window goes through this same offline path,
so comparisons between them are unaffected, but an absolute amplitude here is not the online
`tep_amplitude_raw`. **FAILED** means stop.

A useful sanity identity: for two signals with matched scale, `rel_err ≈ sqrt(2·(1 − corr))`. If the
measured `rel_err` matches that, the correlation explains the whole residual and there is no gain
term left to blame.

In [ ]:
def online_post(rec, state, onset_s):
    """Run the real Preprocessor.preprocess_post on an offline-epoched trial."""
    buf = epoch_raw(rec, onset_s, POST_I_TMIN, POST_I_TMAX)
    if buf is None:
        return None, None
    ts = POST_I_TMIN + np.arange(buf.shape[0]) / RAW_SFREQ
    out = state["pre"].preprocess_post(buf, ts)
    return out, state["pre"].last_post_full


def _saved_pairs(name, limit=None):
    """(saved epoch, onset) for trials whose online post_proc.npy is on disk."""
    d, ev = SESSIONS[name], REC[name]["events"]
    sel = ev[ev.condition.isin(["prime_single_pulse", "predetermined_single"])]
    if limit:
        sel = sel.head(limit)
    out = []
    for r in sel.itertuples():
        fp = d / f"{r.stage}_{int(r.trial_in_stage):04d}_post_proc.npy"
        if fp.is_file():
            out.append((np.load(fp), float(r.onset)))
    return out


LAG_SEARCH = range(-6, 7)        # raw samples; +/-6 at 5 kHz = +/-1.2 ms
SHIFT_PROBE_TRIALS = 20


def calibrate_onset_shift(name):
    """Whole-sample offset between the events.tsv onsets and the online buffers.

    Searched per trial against the saved epochs and taken as the median, because the quantisation
    is a property of the export, not of any one trial.
    """
    rec, state = REC[name], STATE[name]
    pairs = _saved_pairs(name, SHIFT_PROBE_TRIALS)
    if not pairs:
        return 0, np.nan, np.nan
    keep, best = rec.get("onset_shift", 0), []
    curves = []
    for saved, onset in pairs:
        cs = []
        for lag in LAG_SEARCH:
            rec["onset_shift"] = lag
            _, mine = online_post(rec, state, onset)
            cs.append(np.corrcoef(saved.ravel(), mine.ravel())[0, 1]
                      if mine is not None and mine.shape == saved.shape else np.nan)
        curves.append(cs)
        if np.any(np.isfinite(cs)):
            best.append(list(LAG_SEARCH)[int(np.nanargmax(cs))])
    rec["onset_shift"] = keep
    if not best:
        return 0, np.nan, np.nan
    C = np.asarray(curves, dtype=float)
    shift = int(np.median(best))
    at_zero = np.nanmedian(C[:, list(LAG_SEARCH).index(0)])
    at_best = np.nanmedian(C[:, list(LAG_SEARCH).index(shift)])
    agree = float(np.mean(np.asarray(best) == shift))
    print(f"{name}: onset shift search over {len(pairs)} trials -- best lag {shift:+d} samples "
          f"({shift / RAW_SFREQ * 1000:+.2f} ms), chosen by {100*agree:.0f}% of trials; "
          f"median corr {at_zero:.6f} at 0 -> {at_best:.6f} at {shift:+d}")
    return shift, at_zero, at_best


STAGE0, GATE_OK = {}, {}
for name in list(STATE):
    shift, c0, cb = calibrate_onset_shift(name)
    REC[name]["onset_shift"] = shift

    rows = []
    for saved, onset in _saved_pairs(name):
        _, mine = online_post(REC[name], STATE[name], onset)
        if mine is None or mine.shape != saved.shape:
            continue
        a, b = saved.ravel(), mine.ravel()
        rows.append({"corr": float(np.corrcoef(a, b)[0, 1]),
                     "slope": float(np.dot(a, b) / np.dot(b, b)),
                     "rel_err": float(np.linalg.norm(a - b) / np.linalg.norm(a))})
    S = pd.DataFrame(rows)
    STAGE0[name] = S
    if S.empty:
        GATE_OK[name] = None
        print(f"{name}: no overlapping trials with saved post_proc.npy -- gate cannot be evaluated")
        continue

    corr, slope, rel = S["corr"].median(), S.slope.median(), S.rel_err.median()
    print(f"{name}: {len(S)} trials at shift {shift:+d}")
    print(f"   corr    median {corr:.6f}   min {S['corr'].min():.6f}")
    print(f"   slope   median {slope:.6f}   IQR [{S.slope.quantile(.25):.6f}, "
          f"{S.slope.quantile(.75):.6f}]")
    print(f"   rel_err median {rel:.2e}   max {S.rel_err.max():.2e}")

    # rel_err implied by the correlation alone, if there were no scale term at all
    implied = float(np.sqrt(max(0.0, 2 * (1 - corr))))
    scale_off = abs(slope - 1) > 0.01
    if corr > 0.999 and not scale_off:
        GATE_OK[name] = True
        print("   -> GATE PASSED: the offline replay reproduces the online epochs")
    elif corr > 0.999 and scale_off:
        GATE_OK[name] = False
        print(f"   -> GATE FAILED: corr is {corr:.6f} but slope is {slope:.4f}. That is a units "
              f"difference between the .fdt and the online stream. Scale the recording.")
    elif corr >= 0.99 and not scale_off:
        GATE_OK[name] = "usable"
        print(f"   -> USABLE, NOT EXACT. rel_err {rel:.3f} against {implied:.3f} implied by the "
              f"correlation alone, so there is no scale term left to explain -- the residual is "
              f"timing and rounding, not gain.")
        print(f"      A {100*(1-corr):.1f}% variance difference shifts the post-MAD z-score by "
              f"~0.02 against a gate that fires at 6, so trial rejection is unaffected and the "
              f"tables below are not empty because of this.")
        print(f"      Both arms and every candidate baseline window go through this same offline "
              f"path, so comparisons between them are unaffected. What this does limit is any "
              f"claim that an absolute amplitude here equals the online tep_amplitude_raw.")
        if np.isfinite(cb) and cb - c0 > 0.0005:
            print(f"      The shift search already recovered {c0:.5f} -> {cb:.5f}; the rest is "
                  f"sub-sample and would need interpolation to remove.")
    else:
        GATE_OK[name] = False
        print(f"   -> GATE FAILED: corr {corr:.6f} even at the best lag. The epochs are not the "
              f"same samples. Check the .fdt memory order and the event alignment in section 2.")

print()
for name in STATE:
    v = GATE_OK.get(name)
    print(f"  {name}: gate {'passed' if v is True else 'usable (not exact)' if v == 'usable' else 'FAILED' if v is False else 'not evaluated'}"
          f"   onset_shift={REC[name].get('onset_shift', 0):+d} samples")

In [ ]:
# ---------------------------------------------------------------- how many trials actually survive
# Every table below inherits Arm A's accept/reject decision. If preprocess_post rejects nearly
# everything -- a scale mismatch does exactly that -- the tables are not wrong, they are empty,
# and this is where that becomes visible instead of turning into blank cells ten cells later.
for name in STATE:
    ev = REC[name]["events"]
    sel = ev[ev.condition.isin(["prime_single_pulse", "predetermined_single"])]
    kept = sum(online_post(REC[name], STATE[name], float(r.onset))[0] is not None
               for r in sel.head(60).itertuples())
    n = min(60, len(sel))
    print(f"{name}: {kept}/{n} probed single-pulse trials survive preprocess_post "
          f"({100*kept/max(n,1):.0f}%)"
          + ("" if kept >= MIN_N else "  <-- too few; fix this before reading anything below"))

## 5 · An explicit re-implementation of the post pipeline

Arm B has to change one step *inside* `preprocess_post`, and monkey-patching a library is a poor way
to do that. Instead the step sequence is written out here, reading every operator from the session's
own bundle, and then validated: with the original settings it must reproduce
`Preprocessor.preprocess_post()` exactly. Only then is the baseline step varied.

The sequence, in order: crop to [-30, +100] ms and resample → **baseline #1** → interpolate
`artifact_window_1` → interpolate bad channels → average reference → ICA → **baseline #2** → average
reference → crop to [1, 100] ms → SOUND → average reference → SSP-SIR → interpolate [1, 15] ms →
average reference.

The trial-rejection cascade is deliberately *not* re-implemented: Arm A's accept/reject decisions are
inherited, so both arms analyse an identical set of trials and the only difference between them is
the value subtracted as baseline.

**Why the two baseline calls are not the same intervention.** Measured on this data, baseline #1
removes a per-channel constant of order **1.3 × 10⁴** — the raw DC — while baseline #2, applied after
ICA, removes of order **2.5 × 10⁻³**. Seven orders of magnitude apart, and they are doing different
jobs. #1 is gross DC removal, and at that scale a 10 ms mean and a 1 s robust centre agree to within
a fraction of a percent, so changing #1 does almost nothing. #2 is the one operating at the signal's
own scale, which makes it the channel through which a noisy 10-sample estimate reaches the dipole
window. **So "extend the baseline" is really a change to baseline #2.**

- **Arm A** — both baselines as online.
- **Arm B** — long robust centre at **baseline #2**; #1 left local. The version to take seriously.
- **Arm B-naive** — long robust centre at both, the literal reading of the proposal. Included so the
  diagnostic below can show how little the #1 change contributes.

**The implementation requirement that makes or breaks this.** The reference block must sit in the
same frame as the epoch the centre is subtracted from. By the time baseline #2 runs, the epoch has
had #1's constant removed and has been ICA-cleaned; a reference block that skips those steps still
carries the ~1.3 × 10⁴ DC, and subtracting its centre is meaningless — it inflates the finished epoch
by three orders of magnitude. `long_pre_block()` therefore takes #1's own constant as `offset` and
applies the same spatial steps before the centre is taken. The diagnostic prints both baseline
magnitudes per arm so this stays visible rather than silently wrong.

In [ ]:
def _centre(x: np.ndarray, how: str) -> np.ndarray:
    """Per-channel centre of (n_channels, n_times)."""
    if how == "median":
        return np.median(x, axis=1)
    if how == "trimmed":
        return trim_mean(x, 0.1, axis=1)
    return np.mean(x, axis=1)


def long_pre_block(rec, state, onset_s, *, ica_cleaned: bool, offset=None, window=None):
    """(60, n) resampled pre-stimulus block used for the robust centre.

    The block must sit in the same frame as the epoch the centre will be subtracted from. Baseline
    #2 acts on data that has already had baseline #1's constant removed and has been ICA-cleaned, so
    `offset` (baseline #1's per-channel constant) is subtracted first and the same spatial steps are
    then applied. Skipping that leaves the raw ~1e4 DC in the reference and the subtraction is
    meaningless -- the diagnostic below is what makes that visible.
    """
    bundle, info = state["bundle"], state["pre"]._info_processed
    win = tuple(ARM_B_WINDOW if window is None else window)
    raw = epoch_raw(rec, onset_s, win[0], win[1])
    if raw is None:
        return None
    rts = win[0] + np.arange(raw.shape[0]) / RAW_SFREQ
    d = resample_buffer_polyphase(crop_eeg_buffer(raw, rts, *win)[0],
                                  sfreq_from=RAW_SFREQ, sfreq_to=PROC_SFREQ).T
    if offset is not None:
        d = d - np.asarray(offset)[:, None]
    if not ica_cleaned:
        return d
    ep = mne.EpochsArray(d.reshape(1, *d.shape), info, tmin=win[0], verbose=False)
    if bundle["bad_channels"]:
        ep, _ = _interpolate_bad_channels(ep, bundle["bad_channels"],
                                          bundle["channel_interpolation_info"])
    ep.set_eeg_reference("average", projection=False, verbose=False)
    bundle["ica"].apply(ep)
    ep.set_eeg_reference("average", projection=False, verbose=False)
    return ep.get_data(copy=True)[0]


def post_pipeline(rec, state, onset_s, *, baseline_stage="none", robust=ROBUST_CENTRE,
                  baseline_window=None, baseline_centre="mean", long_window=None,
                  artifact_window=None, raw_hold=None,
                  do_baseline=True, do_artifact_interp=True, report=None):
    """Explicit re-implementation of `preprocess_post`.

    Two independent knobs:

    * `baseline_window` -- the window the baseline is estimated over. `None` means the online
      `CFG.baseline` = [-25, -15] ms. This is what section 7b sweeps. `baseline_centre` picks the
      estimator over it ("mean" reproduces `apply_baseline`; "median"/"trimmed" are available for
      the same window).
    * `baseline_stage`  -- which of the two baseline calls is replaced by a robust centre taken
      from `long_window`: "none" (Arm A), "second" (Arm B), "both" (Arm B-naive), "first".

    **The epoch is only [-30, +100] ms.** `Epochs.apply_baseline` therefore cannot reach a window
    starting before -30 ms, and MNE raises rather than silently clipping. Any such window is routed
    through `long_pre_block` instead -- a separate slice of the recording, put in the epoch's own
    frame, whose centre is subtracted as a per-channel constant at exactly the point
    `apply_baseline` would have acted. That is the same machinery Arm B uses, so an "earlier
    baseline window" and "Arm B's robust centre" differ only in extent and estimator, not in
    mechanism. Which route each call took is reported as `route1`/`route2`.

    With every knob left at its default this reproduces `Preprocessor.preprocess_post` exactly,
    which the validation cell below asserts.
    """
    bundle, info = state["bundle"], state["pre"]._info_processed
    bl_win = tuple(CFG.baseline if baseline_window is None else baseline_window)
    art_win = tuple(CFG.artifact_window_1 if artifact_window is None else artifact_window)
    buf = epoch_raw(rec, onset_s, POST_I_TMIN, POST_I_TMAX)
    if buf is None:
        return None
    ts = POST_I_TMIN + np.arange(buf.shape[0]) / RAW_SFREQ

    # Hold the artifact at the local pre-pulse level *before* the resample. The polyphase
    # anti-alias filter is symmetric, so the pulse leaks backwards into the samples a baseline
    # window would average over; flattening it at 5 kHz removes that at the source instead of
    # relying on fix_stim_artifact to zero it afterwards. Held, not zeroed: the raw slice still
    # carries the ~1.3e4 DC and a step to zero would be a bigger discontinuity than the artifact.
    if raw_hold is not None:
        lo, hi = raw_hold
        m = (ts >= lo) & (ts <= hi)
        ref = (ts >= lo - 0.005) & (ts < lo)
        if m.any() and ref.any():
            buf = buf.copy()
            buf[m] = buf[ref].mean(axis=0, keepdims=True)
    b, _ = crop_eeg_buffer(buf, ts, POST_I_TMIN, POST_I_TMAX)
    b = resample_buffer_polyphase(b, sfreq_from=RAW_SFREQ, sfreq_to=PROC_SFREQ)
    ep = _numpy_to_epochs_array(b, info, POST_I_TMIN)

    b1_const = {}

    def apply_bl(epochs, which):
        """which: 1 or 2. Returns the epochs and records the constant actually removed."""
        if not do_baseline:
            return epochs
        arm_long = baseline_stage == "both" or \
                   (baseline_stage == "first" and which == 1) or \
                   (baseline_stage == "second" and which == 2)
        fits = (bl_win[0] >= epochs.times[0] - 1e-9) and (bl_win[1] <= epochs.times[-1] + 1e-9)
        # the arm's own robust centre wins; otherwise a window that does not fit the epoch is
        # still honoured, via the same block route but with the plain estimator asked for
        if arm_long:
            win, est = (tuple(ARM_B_WINDOW if long_window is None else long_window), robust)
        elif not fits:
            win, est = bl_win, baseline_centre
        else:
            win, est = None, None
        before = epochs.get_data(copy=True)[0]
        if win is not None:
            blk = long_pre_block(rec, state, onset_s, ica_cleaned=(which == 2),
                                 offset=b1_const.get("v") if which == 2 else None,
                                 window=win)
            if blk is None:
                return epochs
            d = epochs.get_data(copy=True)
            d[0] -= _centre(blk, est)[:, None]
            epochs = mne.EpochsArray(d, epochs.info, events=epochs.events,
                                     tmin=epochs.times[0], verbose=False)
        elif baseline_centre == "mean":
            epochs = epochs.apply_baseline(bl_win)
        else:
            tm = epochs.times
            m = (tm >= bl_win[0] - 1e-9) & (tm <= bl_win[1] + 1e-9)
            d = epochs.get_data(copy=True)
            d[0] -= _centre(d[0][:, m], baseline_centre)[:, None]
            epochs = mne.EpochsArray(d, epochs.info, events=epochs.events,
                                     tmin=epochs.times[0], verbose=False)
        if report is not None:
            report[f"route{which}"] = ("arm-robust" if arm_long
                                       else "block" if win is not None else "in-epoch")
        # whatever was removed, read back off the data -- correct for either branch
        removed = (before - epochs.get_data(copy=False)[0])[:, 0]
        if which == 1:
            b1_const["v"] = removed
        if report is not None:
            report[f"b{which}"] = float(np.abs(removed).mean())
        return epochs

    ep = apply_bl(ep, 1)
    if do_artifact_interp:
        ep = mne.preprocessing.fix_stim_artifact(
            ep, tmin=art_win[0], tmax=art_win[1], mode="window")
    if bundle["bad_channels"]:
        ep, _ = _interpolate_bad_channels(ep, bundle["bad_channels"],
                                          bundle["channel_interpolation_info"])
    ep.set_eeg_reference("average", projection=False, verbose=False)
    bundle["ica"].apply(ep)
    ep = apply_bl(ep, 2)
    ep.set_eeg_reference("average", projection=False, verbose=False)
    ep.crop(POST_TMIN, POST_TMAX, include_tmax=True)

    d = np.matmul(bundle["sound_filter"], ep.get_data(copy=True))
    ep = mne.EpochsArray(d, ep.info, events=ep.events, tmin=ep.times[0], verbose=False)
    ep.set_eeg_reference("average", projection=False, verbose=False)

    d = ssp_sir_single_trial(ep.get_data(copy=True)[0, :, :],
                             bundle["sspsir_suppression_matrix_P"],
                             bundle["sspsir_sir_projmat_suppr"],
                             bundle["sspsir_sir_projmat_orig"],
                             bundle["sspsir_filter_kernel"])
    ep = mne.EpochsArray(d.reshape(1, *d.shape), ep.info, events=ep.events,
                         tmin=ep.times[0], verbose=False)
    if do_artifact_interp:
        ep = mne.preprocessing.fix_stim_artifact(
            ep, tmin=POST_TMIN, tmax=CFG.artifact_window_2[1], mode="window")
    ep.set_eeg_reference("average", projection=False, verbose=False)
    return ep.get_data(copy=True)[0]

In [ ]:
# ---- validate the re-implementation against the library, original settings ----
for name, rec in REC.items():
    if name not in STATE:
        continue
    ev = rec["events"]
    probe = ev[ev.condition.isin(["prime_single_pulse", "predetermined_single"])].head(30)
    errs = []
    for r in probe.itertuples():
        _, ref = online_post(rec, STATE[name], float(r.onset))
        mine = post_pipeline(rec, STATE[name], float(r.onset), baseline_stage="none")
        if ref is None or mine is None or ref.shape != mine.shape:
            continue
        errs.append(np.linalg.norm(ref - mine) / np.linalg.norm(ref))
    errs = np.asarray(errs)
    print(f"{name}: re-implementation vs Preprocessor.preprocess_post -- "
          f"median rel_err {np.median(errs):.2e}, max {errs.max():.2e}  (n={len(errs)})")
    print(f"   -> {'MATCH, safe to vary the baseline' if errs.max() < 1e-9 else 'DIFFERS -- fix before continuing'}")

In [ ]:
# ---------------------------------------------------------------- what each baseline actually removes
ARMS = {"A (online)": "none", "B (long @ #2)": "second", "B-naive (long @ both)": "both"}

for name, rec in REC.items():
    if name not in STATE:
        continue
    sel = rec["events"]
    sel = sel[sel.condition.isin(["prime_single_pulse", "predetermined_single"])].head(20)
    rows = []
    for label, stage in ARMS.items():
        b1, b2, mag = [], [], []
        for r in sel.itertuples():
            rep = {}
            full = post_pipeline(rec, STATE[name], float(r.onset),
                                 baseline_stage=stage, report=rep)
            if full is None:
                continue
            b1.append(rep.get("b1", np.nan)); b2.append(rep.get("b2", np.nan))
            mag.append(np.abs(full).mean())
        rows.append({"arm": label, "|baseline #1|": np.nanmedian(b1),
                     "|baseline #2|": np.nanmedian(b2),
                     "|final epoch|": np.nanmedian(mag)})
    T = pd.DataFrame(rows).set_index("arm")
    print(f"--- {name} ---")
    display(T)
    a = T.loc["A (online)", "|final epoch|"]
    print(f"  final-epoch magnitude relative to Arm A: "
          f"B={T.loc['B (long @ #2)','|final epoch|']/a:.3f}x, "
          f"B-naive={T.loc['B-naive (long @ both)','|final epoch|']/a:.3f}x")
    print(f"  Baseline #1 removes ~{T.loc['A (online)','|baseline #1|']:,.0f} (the raw DC) in every arm --")
    print(f"  at that scale a 10 ms mean and a 1 s robust centre agree, so the #1 change does little.")
    print(f"  The substantive change is at #2: {T.loc['A (online)','|baseline #2|']:.2e} online vs "
          f"{T.loc['B (long @ #2)','|baseline #2|']:.2e} robust.")
    print(f"  If |baseline #2| for an Arm B row came out near {T.loc['A (online)','|baseline #1|']:,.0f}, the")
    print(f"  reference block was not put in the epoch's frame -- see long_pre_block's offset argument.")

## 6 · The pre-stimulus dipole floor

To ask "how much dipole-space amplitude is there anyway", the pre-stimulus period has to be pushed
through machinery *geometrically identical* to a post epoch. That is not optional bookkeeping:
`sspsir_filter_kernel` has shape (60, 100), i.e. it is a temporal filter sized to the 100-sample post
window, so it cannot be applied to a 1 s block in one go.

The pre-stimulus second is therefore cut into overlapping **100 ms chunks**, each shaped exactly like
a post epoch, each run through the same operators, and each fitted with the same dipole at the same
`index_range`. That also keeps the number of comparison windows modest (~90 rather than ~1000),
which matters for the maximum in section 7.

In [ ]:
N_DIP = epoch_n_times(DIPOLE_TMIN, DIPOLE_TMAX, PROC_SFREQ)
N_POST = epoch_n_times(POST_TMIN, POST_TMAX, PROC_SFREQ)
DIP_OFFSET = int(round((DIPOLE_TMIN - POST_TMIN) * PROC_SFREQ))   # dipole window start within the post epoch


def fit_amp(state, epoch_dip: np.ndarray, orientation: str) -> float:
    fi = state["fitter"]._fitting_info
    ori = fi["orientation"] if orientation == "fixed" else None
    return float(fit_dipole_to_single_trial(
        epoch_dip, state["fitter"]._forward, fi["position_index"], ori, fi["index_range"]))


# The post epoch has its per-channel DC removed by baseline #1 before anything spatial happens.
# A pre-stimulus chunk must get the same treatment or it is not comparable: an average reference
# removes what is common across channels, not each channel's own offset, so an unbaselined chunk
# carries the raw ~1.3e4 DC straight into the dipole fit and comes out ~1000x a post amplitude.
# Geometrically, the online baseline sits 16-26 ms before the start of the 100-sample window the
# dipole is fitted in, so that is the offset used here.
BL_LEAD_HI = int(round(0.026 * PROC_SFREQ))   # baseline starts this many samples before a chunk
BL_LEAD_LO = int(round(0.016 * PROC_SFREQ))   # ...and ends this many before it
CHUNK_LEAD = BL_LEAD_HI + 4                   # lead-in the block must carry


def _chunk_baselined(d: np.ndarray, s: int) -> np.ndarray:
    """(60, N_POST) chunk starting at column `s` of block `d`, with its own local baseline gone."""
    bl = d[:, s - BL_LEAD_HI:s - BL_LEAD_LO].mean(axis=1)
    return d[:, s:s + N_POST] - bl[:, None]


def pre_floor(rec, state, onset_s, *, baseline_stage="none", robust=ROBUST_CENTRE,
              step_ms=PRE_CHUNK_STEP_MS, orientation="fixed", span=None):
    """Dipole amplitudes from post-epoch-shaped chunks tiling the pre-stimulus period.

    `orientation` may be a tuple, in which case a dict keyed by orientation comes back and the
    ~90 chunk pipelines are run once instead of once per convention. Fitting is the cheap part;
    ICA + SOUND + SSP-SIR per chunk is not.
    """
    multi = not isinstance(orientation, str)
    oris = tuple(orientation) if multi else (orientation,)
    bundle, info = state["bundle"], state["pre"]._info_processed
    lo, hi = tuple(PRE_FLOOR_SPAN if span is None else span)
    lead = CHUNK_LEAD / PROC_SFREQ
    raw = epoch_raw(rec, onset_s, lo - lead, hi)
    if raw is None:
        return {o: np.array([]) for o in oris} if multi else np.array([])
    rts = (lo - lead) + np.arange(raw.shape[0]) / RAW_SFREQ
    d = resample_buffer_polyphase(*crop_eeg_buffer(raw, rts, lo - lead, hi)[:1],
                                  sfreq_from=RAW_SFREQ, sfreq_to=PROC_SFREQ).T  # (60, n)

    if baseline_stage != "none":
        d = d - _centre(d, robust)[:, None]

    amps, step = {o: [] for o in oris}, max(1, int(round(step_ms)))
    for s in range(CHUNK_LEAD, d.shape[1] - N_POST + 1, step):
        chunk = _chunk_baselined(d, s)
        ep = mne.EpochsArray(chunk.reshape(1, *chunk.shape), info, tmin=POST_TMIN, verbose=False)
        if bundle["bad_channels"]:
            ep, _ = _interpolate_bad_channels(ep, bundle["bad_channels"],
                                              bundle["channel_interpolation_info"])
        ep.set_eeg_reference("average", projection=False, verbose=False)
        bundle["ica"].apply(ep)
        ep.set_eeg_reference("average", projection=False, verbose=False)
        x = np.matmul(bundle["sound_filter"], ep.get_data(copy=True))[0]
        x = x - x.mean(axis=0, keepdims=True)
        x = ssp_sir_single_trial(x, bundle["sspsir_suppression_matrix_P"],
                                 bundle["sspsir_sir_projmat_suppr"],
                                 bundle["sspsir_sir_projmat_orig"],
                                 bundle["sspsir_filter_kernel"])
        x = x - x.mean(axis=0, keepdims=True)
        dip = x[:, DIP_OFFSET:DIP_OFFSET + N_DIP]
        for o in oris:
            amps[o].append(fit_amp(state, dip, o))
    if multi:
        return {o: np.asarray(v) for o, v in amps.items()}
    return np.asarray(amps[oris[0]])


# sanity: the post-window fit routed through fit_amp must equal the library's fit_trial
for name, rec in REC.items():
    if name not in STATE:
        continue
    ev = rec["events"]
    r = ev[ev.condition == "predetermined_single"].iloc[0]
    post, _ = online_post(rec, STATE[name], float(r.onset))
    if post is not None:
        a = STATE[name]["fitter"].fit_trial(post)
        b = fit_amp(STATE[name], post, "free")
        print(f"{name}: fit_trial={a:.8f}  fit_amp(free)={b:.8f}  "
              f"-> {'identical' if abs(a-b) < 1e-12 else 'DIFFER'}")

## 7 · Per-trial measures, both arms

For every single-pulse intervention trial, under Arm A (both baselines as online) and Arm B (1 s
robust centre at baseline #2), and under both orientation conventions:

- `post_amp` — the dipole amplitude in the real [38, 50] ms window.
- `pre_max`, `pre_med`, `pre_sd` — over the pre-stimulus chunks.
- `diff = post_amp - pre_max` — the difference as originally specified.
- `pctile` — where `post_amp` sits in that trial's own pre-stimulus distribution, and `z`, its
  standardised equivalent. These are the headline: `pre_max` is a maximum over ~90 windows while
  `post_amp` is a single draw, so `diff` is biased downward and noisy by construction, whereas the
  percentile compares like with like.

Fixed orientation projects onto the calibrated dipole and is signed; free orientation returns a norm
and is therefore strictly positive, which biases any difference. Fixed is the headline; both are
computed so the choice can be checked rather than argued.

In [ ]:
def trial_table(name, *, arms=(("A", "none"), ("B", "second")),
                orientations=("fixed", "free"), limit=None):
    rec, state = REC[name], STATE[name]
    ev = rec["events"]
    sel = ev[ev.condition.isin(["prime_single_pulse", "predetermined_single"])]
    if limit:
        sel = sel.head(limit)
    rows = []
    for r in sel.itertuples():
        onset = float(r.onset)
        keep, _ = online_post(rec, state, onset)     # Arm A decides inclusion for both arms
        if keep is None:
            continue
        row = {"stage": r.stage, "trial_in_stage": int(r.trial_in_stage),
               "condition": r.condition,
               "tep_amplitude_raw": getattr(r, "tep_amplitude_raw", np.nan),
               "tep_amplitude": getattr(r, "tep_amplitude", np.nan),
               "prediction_probability": getattr(r, "prediction_probability", np.nan)}
        for arm, stage in arms:
            full = post_pipeline(rec, state, onset, baseline_stage=stage)
            if full is None:
                continue
            dip = full[:, DIP_OFFSET:DIP_OFFSET + N_DIP]
            # one pass over the ~90 pre-stimulus chunks, both orientations fitted from it
            floors = pre_floor(rec, state, onset, baseline_stage=stage,
                               orientation=tuple(orientations))
            for ori in orientations:
                p = fit_amp(state, dip, ori)
                floor = floors[ori]
                tag = f"{arm}_{ori}"
                row[f"post_{tag}"] = p
                if floor.size:
                    row[f"premax_{tag}"] = floor.max()
                    row[f"premed_{tag}"] = np.median(floor)
                    row[f"diff_{tag}"] = p - floor.max()
                    row[f"pctile_{tag}"] = 100.0 * (floor < p).mean()
                    row[f"z_{tag}"] = (p - floor.mean()) / (floor.std() or np.nan)
        rows.append(row)
    return pd.DataFrame(rows)


# Start small: this is the expensive cell. Raise/remove `limit` once it looks right.
TRIALS = {}
for name in STATE:
    TRIALS[name] = trial_table(name, limit=40)
    print(f"{name}: {len(TRIALS[name])} trials")
    display(TRIALS[name].head())

## 8 · Alignment with the online record

`tep_amplitude_raw` is the online dipole amplitude on the same scale this notebook computes;
`tep_amplitude` is the CDF-normalised [0, 1] label produced by `TEPNormalizer` and is *not*
comparable in magnitude. Arm A's `post_A_fixed`/`post_A_free` should track `tep_amplitude_raw` closely —
that is the second half of the replication gate. Arm B will not, by design.

In [ ]:
for name, T in TRIALS.items():
    print(f"--- {name} ---")
    if T.empty:
        print("  no trials in the table; see the survival count above")
        continue
    for col in [c for c in T.columns if c.startswith(("post_", "diff_", "pctile_", "z_"))]:
        d = T[[col, "tep_amplitude_raw"]]
        r, n, why = safe_corr(d[col], d.tep_amplitude_raw)
        if why:
            print(f"  {col:22s} vs tep_amplitude_raw: -- ({why})")
            continue
        extra = ""
        if col.startswith("post_"):
            j = d.dropna()                       # same rows on both sides, always
            j = j[j.tep_amplitude_raw != 0]
            extra = (f"  median ratio={np.median(j[col] / j.tep_amplitude_raw):.4f}"
                     if len(j) else "  median ratio=--")
        print(f"  {col:22s} vs tep_amplitude_raw: r={r:+.4f}{extra}  n={n}")

## 9 · Which baseline window?

The section above contrasts two arms. This one turns the baseline window itself into the variable,
because that is the decision the config actually has to make.

Each candidate is scored against a **fixed pre-stimulus probe** — a post-epoch-shaped window at
`PROBE_LAG_MS`, carrying its own local baseline but untouched by the candidate under test, so it is
byte-identical across the sweep. `r(post, pre-probe)` is then a clean read of how much pre-stimulus
amplitude the baseline choice lets through into the dipole window. Lower is better, but only if
`median |post|` does not collapse with it — a baseline that destroys the response also destroys the
coupling.

Three constraints bound the choice, and the setup cell prints where each candidate sits against all
three:

- the classifier reads **[-65, -15] ms**, so a baseline inside that is subtracted from the very
  epoch the classifier is asked to predict;
- `fix_stim_artifact` tapers everything after **-14 ms** toward zero, so a window later than that is
  near-empty by the time baseline #2 runs;
- the post epoch is only **[-30, +100] ms**, so `apply_baseline` cannot reach anything earlier than
  -30 ms at all. Those windows are routed through `long_pre_block` instead — the same machinery
  Arm B uses — which is why "use an earlier baseline" and "use Arm B's robust centre" are the same
  intervention with different extents.

`predetermined_single` trials fired on a fixed clock, so they carry no PRIME selection; their column
is the one to read.

In [ ]:
def probe_amplitude(rec, state, onset_s, lag_ms=PROBE_LAG_MS, orientation="fixed"):
    """Dipole amplitude of a post-epoch-shaped window sitting `lag_ms` before the pulse.

    It carries its own local baseline (the same 16-26 ms lead the online epoch uses) so that it is
    on the post epoch's scale, but it is not touched by the candidate baseline window under test --
    so it stays identical across the sweep and any change in its correlation with the post
    amplitude is attributable to the baseline choice alone.
    """
    lo = lag_ms / 1000.0
    lead = CHUNK_LEAD / PROC_SFREQ
    raw = epoch_raw(rec, onset_s, lo - lead, lo + N_POST / PROC_SFREQ)
    if raw is None:
        return np.nan
    rts = (lo - lead) + np.arange(raw.shape[0]) / RAW_SFREQ
    d = resample_buffer_polyphase(*crop_eeg_buffer(raw, rts, lo - lead,
                                                   lo + N_POST / PROC_SFREQ)[:1],
                                  sfreq_from=RAW_SFREQ, sfreq_to=PROC_SFREQ).T
    if d.shape[1] < CHUNK_LEAD + N_POST:
        return np.nan
    chunk = _chunk_baselined(d, CHUNK_LEAD)
    ep = mne.EpochsArray(chunk.reshape(1, *chunk.shape), state["pre"]._info_processed,
                         tmin=POST_TMIN, verbose=False)
    if state["bundle"]["bad_channels"]:
        ep, _ = _interpolate_bad_channels(ep, state["bundle"]["bad_channels"],
                                          state["bundle"]["channel_interpolation_info"])
    ep.set_eeg_reference("average", projection=False, verbose=False)
    state["bundle"]["ica"].apply(ep)
    x = np.matmul(state["bundle"]["sound_filter"], ep.get_data(copy=True))[0]
    x = x - x.mean(axis=0, keepdims=True)
    return fit_amp(state, x[:, DIP_OFFSET:DIP_OFFSET + N_DIP], orientation)


def baseline_sweep(name, *, windows=None, orientation="fixed", limit=None,
                   conditions=("predetermined_single", "prime_single_pulse")):
    """Per-trial post-stimulus amplitude under each candidate baseline window.

    Returns a long table: one row per (trial, window). `probe` is the same baseline-free
    pre-stimulus amplitude for every window of a given trial, so corr(post, probe) isolates what
    the baseline choice does to the pre-to-post link.
    """
    windows = BASELINE_WINDOWS if windows is None else windows
    rec, state = REC[name], STATE[name]
    sel = rec["events"]
    sel = sel[sel.condition.isin(conditions)]
    if limit:
        sel = sel.head(limit)
    rows = []
    for r in sel.itertuples():
        onset = float(r.onset)
        keep, _ = online_post(rec, state, onset)      # inherit Arm A's accept/reject
        if keep is None:
            continue
        probe = probe_amplitude(rec, state, onset, orientation=orientation)
        for lbl, win in windows.items():
            full = post_pipeline(rec, state, onset, baseline_window=win)
            if full is None:
                continue
            rows.append({
                "window": lbl, "lo_ms": win[0] * 1000, "hi_ms": win[1] * 1000,
                "stage": r.stage, "trial_in_stage": int(r.trial_in_stage),
                "condition": r.condition, "probe": probe,
                "post": fit_amp(state, full[:, DIP_OFFSET:DIP_OFFSET + N_DIP], orientation),
                "tep_amplitude_raw": getattr(r, "tep_amplitude_raw", np.nan),
            })
    return pd.DataFrame(rows)


def summarise_sweep(S):
    """corr(post, probe) per window, overall and for the unselected trials alone."""
    if S.empty:
        print("  the sweep produced no rows -- no single-pulse trial survived preprocess_post")
        return pd.DataFrame()
    out, notes = [], []
    for lbl, g in S.groupby("window", sort=False):
        p = g[g.condition == "predetermined_single"]
        r_all, n_all, w_all = safe_corr(g.post, g.probe)
        r_pre, n_pre, w_pre = safe_corr(p.post, p.probe)
        r_raw, n_raw, w_raw = safe_corr(g.post, g.tep_amplitude_raw)
        out.append({"window": lbl, "n": n_all, "median |post|": g.post.abs().median(),
                    "r(post, pre-probe)": r_all, "r, predetermined only": r_pre,
                    "n predet": n_pre, "n vs online": n_raw, "r(post, online raw)": r_raw})
        for col, why in (("r(post, pre-probe)", w_all), ("r, predetermined only", w_pre),
                         ("r(post, online raw)", w_raw)):
            if why:
                notes.append(f"    {lbl:20s} {col:24s} blank: {why}")
    if notes:
        print("  blank cells, and why:")
        print("\n".join(dict.fromkeys(notes)))
    return pd.DataFrame(out).set_index("window")


SWEEPS_BL = {}
for name in STATE:
    SWEEPS_BL[name] = baseline_sweep(name, limit=BASELINE_SWEEP_LIMIT)
    print(f"--- {name} --- ({SWEEPS_BL[name].trial_in_stage.nunique()} trials "
          f"x {SWEEPS_BL[name].window.nunique()} windows)")
    display(summarise_sweep(SWEEPS_BL[name]).round(4))
print("Read the 'r(post, pre-probe)' column: the baseline window that minimises it without")
print("collapsing |post| is the one that couples the two windows least. 'r(post, online raw)'")
print("is a sanity check -- it must be ~1.000 for the online window and only for that one.")

## 9b · Can the artifact window be narrowed so a [-15, -5] baseline becomes usable?

Three measurements on pilot003's raw buffers settle what is and is not in the way.

**The recorded artifact starts at -1.2 ms, not -14.** From -15 to -2 ms the raw 5 kHz signal sits
at 2.7x the quiet level, against 2.4x inside the current baseline window. The hardware artifact is
not what makes [-15, -5] unusable.

**`fix_stim_artifact(mode="window")` zeroes its whole interval**, with a short taper at each edge —
it is not a notch centred on the pulse. Narrowing it to [-4, +14] leaves everything from -2 ms
onward at zero, so the pulse stays fully suppressed, and restores the samples between -14 and -4.
That part of the idea works exactly as intended.

**What blocks it is the order of operations.** `preprocess_post` resamples 5 kHz -> 1 kHz before
the baseline and before the artifact fix, and the polyphase anti-alias filter is symmetric, so the
~30,000x spike at +0.6 ms leaks *backwards*: 2.3x the quiet level at -9 ms, 5.1x at -7, 13.0x at
-5, 20.3x at -4. Today the [-14, +14] taper hides all of that by zeroing it. Narrow the taper and
the smear is exposed inside [-15, -5], whose mean then sits at **3.91x** the quiet level against
1.55x for [-25, -15]. Baselining on it subtracts a pulse-derived constant from every sample of the
epoch, dipole window included — manufacturing precisely the coupling this notebook exists to detect.

Two ways out, both testable below:

- **Hold the artifact at 5 kHz before the resample** (`raw_hold`). Flattening [-2, +15] ms to the
  local pre-pulse level drops [-15, -5] from 3.91x to **1.86x**, the local noise floor. The smear
  is removed at its source, and [-15, -5] becomes a legitimate window that also sits outside the
  classifier's [-65, -15] ms.
- **Or end the baseline before the smear**: `[-15, -10]` is 1.85x as the pipeline already stands,
  clean and outside the model window, at the cost of 6 samples instead of 10.

One caveat for either: the ICA, SOUND, SSP-SIR, ocular thresholds and MAD statistics in the bundle
were all fitted downstream of `artifact_window_1`. Changing it invalidates them, so this is a
recalibration for a future session, not a re-analysis knob for 003/004/005.

In [ ]:
PROPOSALS = {
    "as online":                    dict(),
    "taper [-4,+14] + bl [-15,-5]": dict(artifact_window=(-0.004, 0.014),
                                         baseline_window=(-0.015, -0.005)),
    "   + hold [-2,+15] @ 5 kHz":   dict(artifact_window=(-0.004, 0.014),
                                         baseline_window=(-0.015, -0.005),
                                         raw_hold=(-0.002, 0.015)),
    "bl [-15,-10], taper as is":    dict(baseline_window=(-0.015, -0.010)),
}


def try_proposals(name, limit=60, orientation="fixed"):
    rec, state = REC[name], STATE[name]
    sel = rec["events"]
    sel = sel[sel.condition.isin(["prime_single_pulse", "predetermined_single"])].head(limit)
    post = {k: [] for k in PROPOSALS}
    probe, cond = [], []
    for r in sel.itertuples():
        onset = float(r.onset)
        if online_post(rec, state, onset)[0] is None:
            continue
        probe.append(probe_amplitude(rec, state, onset, orientation=orientation))
        cond.append(r.condition)
        for k, kw in PROPOSALS.items():
            full = post_pipeline(rec, state, onset, **kw)
            post[k].append(np.nan if full is None else
                           fit_amp(state, full[:, DIP_OFFSET:DIP_OFFSET + N_DIP], orientation))
    probe = np.asarray(probe); cond = np.asarray(cond)
    pre_only = cond == "predetermined_single"
    rows = []
    for k, v in post.items():
        v = np.asarray(v)
        r_all, n_all, _ = safe_corr(v, probe)
        r_pre, n_pre, _ = safe_corr(v[pre_only], probe[pre_only])
        rows.append({"variant": k, "n": n_all, "median |post|": np.nanmedian(np.abs(v)),
                     "r(post, pre-probe)": r_all,
                     "r, predetermined only": r_pre, "n predet": n_pre})
    return pd.DataFrame(rows).set_index("variant")


for name in STATE:
    print(f"--- {name} ---")
    display(try_proposals(name).round(4))
print("A variant is an improvement only if r drops while median |post| holds up. If holding the")
print("artifact at 5 kHz drops r but the bare taper change does not, the smear was the mechanism.")

## 10 · Diagnostic 1 — the lag sweep

This is the test that distinguishes the two explanations, and it is the reason the notebook exists.

Slide the pre-stimulus reference window away from the stimulus and measure how strongly it couples to
the post-stimulus dipole amplitude at each lag. The two candidate sources have different signatures:

- **Mechanical leakage** (short baseline, `filtfilt` smearing, artifact interpolation) acts over the
  filter's own timescale — tens of ms. It produces a sharp rise as the window approaches t=0 and
  nothing beyond it.
- **Physiological carryover** — ongoing mu/alpha state, the thing PRIME is built to exploit — decays
  smoothly over hundreds of ms and may oscillate.

If Arm A shows a short-lag spike that Arm B flattens, the baseline was the culprit and the fix is a
config change. If the spike survives Arm B, it is the filter.

In [ ]:
LAGS_MS = [-1000, -800, -600, -400, -300, -200, -150, -100, -70, -50, -40, -35]


def lag_sweep(name, stage="none", orientation="fixed", limit=60):
    """corr(reference-window amplitude, post-stimulus amplitude) as the reference slides away.

    The reference windows come from `probe_amplitude`, so they carry the same local baseline the
    online epoch does. Before that was fixed they carried the raw per-channel DC instead, which an
    average reference does not remove, and this sweep measured DC drift rather than brain.
    """
    rec, state = REC[name], STATE[name]
    ev = rec["events"]
    sel = ev[ev.condition.isin(["prime_single_pulse", "predetermined_single"])].head(limit)
    post_amp, lag_amp = [], {L: [] for L in LAGS_MS}
    for r in sel.itertuples():
        onset = float(r.onset)
        keep, _ = online_post(rec, state, onset)
        if keep is None:
            continue
        full = post_pipeline(rec, state, onset, baseline_stage=stage)
        if full is None:
            continue
        post_amp.append(fit_amp(state, full[:, DIP_OFFSET:DIP_OFFSET + N_DIP], orientation))
        for L in LAGS_MS:
            lag_amp[L].append(probe_amplitude(rec, state, onset, lag_ms=L,
                                              orientation=orientation))
    post_amp = np.asarray(post_amp)
    if post_amp.size == 0:
        print(f"   {name}: no trial survived preprocess_post -- nothing to sweep")
        return pd.DataFrame(columns=["lag_ms", "n", "r", "note"])
    out = []
    for L in LAGS_MS:
        r, n, why = safe_corr(post_amp, np.asarray(lag_amp[L]))
        out.append({"lag_ms": L, "n": n, "r": r, "note": why})
    return pd.DataFrame(out)


SWEEPS = {}
for name in STATE:
    for arm, stage in (("A", "none"), ("B", "second")):
        SWEEPS[(name, arm)] = lag_sweep(name, stage=stage)
        print(f"{name} / Arm {arm}:")
        print(SWEEPS[(name, arm)].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
for (name, arm), S in SWEEPS.items():
    c = [COLOR_PRIME_SINGLE, COLOR_PREDETERMINED, "#5ad8a6"][list(STATE).index(name) % 3]
    ax.plot(S.lag_ms, S.r, "o-" if arm == "A" else "s--", color=c, lw=1.6,
            alpha=1.0 if arm == "A" else 0.55,
            label=f"{name} — {'Arm A (10 ms baseline)' if arm=='A' else 'Arm B (1 s robust @ #2)'}")
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("lag of the reference window relative to the pulse (ms)")
ax.set_ylabel("corr(reference-window amplitude, post-stimulus amplitude)")
ax.set_title("Lag sweep — a short-lag spike means leakage, a smooth decay means physiology")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 11 · Diagnostic 2 — step ablation

The lag sweep says *whether* the coupling is mechanical; this says *which step* makes it. Each
variant disables exactly one thing and the coupling is remeasured against a fixed reference window
just before the stimulus, where leakage should be strongest.

Note the filter cannot be toggled from here — it lives inside `resample_buffer_polyphase` and the
calibration-time filtering rather than in this epoch path — so the causal-filter comparison is listed
as the follow-up it is, not silently skipped.

In [ ]:
VARIANTS = {
    "as online":            dict(baseline_stage="none",   do_baseline=True,  do_artifact_interp=True),
    "long robust @ #2":     dict(baseline_stage="second", do_baseline=True,  do_artifact_interp=True),
    "long robust @ both":   dict(baseline_stage="both",   do_baseline=True,  do_artifact_interp=True),
    "no baseline at all":   dict(baseline_stage="none",   do_baseline=False, do_artifact_interp=True),
    "no artifact interp":   dict(baseline_stage="none",   do_baseline=True,  do_artifact_interp=False),
    "neither":              dict(baseline_stage="none",   do_baseline=False, do_artifact_interp=False),
}

def ablate(name, limit=60, orientation="fixed", ref_lag_ms=-35):
    rec, state = REC[name], STATE[name]
    sel = rec["events"]
    sel = sel[sel.condition.isin(["prime_single_pulse", "predetermined_single"])].head(limit)
    ref, per_variant = [], {k: [] for k in VARIANTS}
    for r in sel.itertuples():
        onset = float(r.onset)
        keep, _ = online_post(rec, state, onset)
        if keep is None:
            continue
        ref.append(probe_amplitude(rec, state, onset, lag_ms=ref_lag_ms,
                                   orientation=orientation))
        for k, kw in VARIANTS.items():
            full = post_pipeline(rec, state, onset, **kw)
            per_variant[k].append(
                np.nan if full is None
                else fit_amp(state, full[:, DIP_OFFSET:DIP_OFFSET + N_DIP], orientation))
    ref = np.asarray(ref)
    rows = []
    for k, v in per_variant.items():
        r, n, why = safe_corr(ref, np.asarray(v))
        rows.append({"variant": k, "n": n, "corr with pre-stim reference": r, "note": why})
    return pd.DataFrame(rows).set_index("variant")


for name in STATE:
    print(f"--- {name} ---")
    display(ablate(name).round(4))
print("A variant whose correlation collapses toward zero identifies the step responsible.")

## 12 · Reading the result

The decision table, to be filled in from the cells above rather than assumed:

| lag sweep (Arm A) | Arm B effect | ablation | reading |
|---|---|---|---|
| short-lag spike | flattens it | dies without baseline | the 10 ms baseline is the culprit; a one-line config change fixes it |
| short-lag spike | survives | dies without artifact interp | interpolation across the stimulus; widen or handle it causally |
| short-lag spike | survives | survives everything | `filtfilt` smearing; needs a causal filter, which is the follow-up below |
| smooth decay over hundreds of ms | little effect | little effect | genuine ongoing-state carryover — PRIME's premise, not an artifact |

That last row is the one to be careful about. **If the coupling is physiological, subtracting a
pre-stimulus floor removes exactly the signal PRIME is built to exploit**, and the proposed change
would make the predictor's job harder rather than fairer. The diagnostics are what separate the two
cases; the difference metric on its own cannot.

Two caveats that hold regardless of outcome. Trial selection is inherited from Arm A throughout, so
the arms are comparable to each other but both inherit whatever selection the online rejection
cascade imposed — including the fact that `reject_range` overlaps the dipole window. And
`prime_single_pulse` trials were *selected* on a high pre-stimulus probability while
`predetermined_single` trials fired on a fixed clock, so any coupling in the predetermined arm cannot
come from PRIME's own selection; that arm is the cleaner read, which is why every table splits by
condition.

## Follow-up this notebook does not do

The causal-filter comparison. `filter_opts` is applied with `filtfilt` well upstream of the epoch
path re-implemented here, so testing it means re-running calibration with a causal (`lfilter`)
variant and comparing — a larger job, and one worth doing only if section 10 points at the filter.

In [ ]:
print("Checklist before trusting anything above:")
for name in SESSIONS:
    parts = []
    parts.append(f"recording loaded={name in REC}")
    if name in STAGE0 and len(STAGE0[name]):
        S = STAGE0[name]
        parts.append(f"stage-0 corr={S['corr'].median():.5f}")
        parts.append(f"slope={S.slope.median():.4f}")
    parts.append(f"gate={GATE_OK.get(name)}")
    parts.append(f"state restored={name in STATE}")
    parts.append(f"trials={len(TRIALS.get(name, []))}")
    print(f"  {name}: " + ", ".join(parts))
print()
print("Reading a gate verdict:")
print("  passed              -> the offline replay is the online epoch")
print("  usable (not exact)  -> a sub-sample timing residue remains. Arms and baseline windows all")
print("                         share this path, so comparisons hold; absolute amplitudes are not")
print("                         the online tep_amplitude_raw.")
print("  FAILED, slope != 1  -> units differ between the .fdt and the online stream")
print("  FAILED, corr low    -> not the same samples; check .fdt memory order and section 2")
print()
print("If a table below is blank, read in this order: the gate verdict, the survival count above")
print("it, then the 'blank cells, and why' lines printed with the table.")